In [ ]:
pip install tensorflow keras numpy matplotlib opencv-python


In [ ]:
import os
from PIL import Image
import numpy as np

# Define the base directory for the dataset
dataset_base_dir = 'dataset'

# Define the subdirectories for training and validation
subdirs = ['train', 'val']

# Define the class names (categories) for our dummy data
class_names = ['healthy', 'diseased']

# Number of dummy images per class per split
num_dummy_images = 5

print(f"Creating dummy dataset structure in: {os.getcwd()}/{dataset_base_dir}")

for subdir in subdirs:
    for class_name in class_names:
        # Create the directory path: dataset/train/healthy, dataset/val/diseased, etc.
        path = os.path.join(dataset_base_dir, subdir, class_name)
        os.makedirs(path, exist_ok=True)
        print(f"Created directory: {path}/")

        # Create dummy image files
        for i in range(num_dummy_images):
            # Create a dummy image (e.g., a white image) and save it
            dummy_image_path = os.path.join(path, f'dummy_image_{i+1}.png')
            img = Image.fromarray(np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8))
            img.save(dummy_image_path)
            # print(f"  Created dummy image: {dummy_image_path}")

print("\nDummy dataset structure created successfully!")


Creating dummy dataset structure in: /content/dataset
Created directory: dataset/train/healthy/
Created directory: dataset/train/diseased/
Created directory: dataset/val/healthy/
Created directory: dataset/val/diseased/

Dummy dataset structure created successfully!


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# Load dataset
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/train",
    image_size=(128, 128),
    batch_size=32
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/val",
    image_size=(128, 128),
    batch_size=32
)

# Build CNN model
# Note: Adjusted the output dense layer to match the number of classes identified by train_data
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(train_data.class_names), activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

# Save model
model.save("plant_disease_model.keras") # Changed to save as .keras

Found 10 files belonging to 2 classes.
Found 10 files belonging to 2 classes.
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6891 - val_accuracy: 0.5000 - val_loss: 2.4982
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 814ms/step - accuracy: 0.5000 - loss: 2.4610 - val_accuracy: 0.5000 - val_loss: 0.7190
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.6810 - val_accuracy: 0.5000 - val_loss: 0.7676
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 718ms/step - accuracy: 0.5000 - loss: 0.7518 - val_accuracy: 0.5000 - val_loss: 0.6921
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 504ms/step - accuracy: 0.8000 - loss: 0.6787 - val_accuracy: 0.5000 - val_loss: 0.7224
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 564ms/step - accuracy: 0.5000 - loss: 0.6946 - val_accuracy: 0.5000 - val_loss: 0.6964
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 686ms/step - accuracy: 0.5000 - loss: 0.6531 - val_accuracy: 0.5000 - val_loss: 0.7073
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 451ms/step - accuracy: 0.5000 - loss: 0.6534 - val_accuracy: 0.5000 - val_loss: 0.6976
Epoch 9/1

In [ ]:
import numpy as np
import cv2
from tensorflow.keras.models import load_model
import os
from PIL import Image # Added for creating dummy image
import zipfile # Added for diagnosis

# Define the model path, using the recommended .keras format
model_filename = "plant_disease_model.keras" # Changed to .keras

# --- DIAGNOSTIC START ---
print(f"Attempting to load model from: {model_filename}")
if os.path.exists(model_filename):
    print(f"'{model_filename}' exists. File size: {os.path.getsize(model_filename)} bytes.")
    if os.path.getsize(model_filename) == 0:
        print("Warning: The model file is empty. It might not have been saved correctly by the previous cell.")
    else:
        try:
            with zipfile.ZipFile(model_filename, 'r') as zf:
                print(f"'{model_filename}' appears to be a valid zip file. Contents: {zf.namelist()}")
        except zipfile.BadZipFile:
            print(f"Error: '{model_filename}' is not a valid zip file. It might be corrupted or not a proper .keras format.")
        except Exception as e:
            print(f"Error checking '{model_filename}' as zip file: {e}")
else:
    print(f"Error: '{model_filename}' does not exist. Please ensure the model training cell was run successfully.")
# --- DIAGNOSTIC END ---

# Load model
try:
    model = load_model(model_filename)
except Exception as e:
    print(f"Error loading model '{model_filename}': {e}")
    print("Please ensure the model was saved correctly as a .keras file in the previous cell.")
    # Exit or handle the error appropriately, for now we'll let it raise if it fails
    raise

# Create a dummy image for prediction if 'test.jpg' doesn't exist
test_image_path = "test.jpg"
if not os.path.exists(test_image_path):
    print(f"Warning: '{test_image_path}' not found. Creating a dummy image for prediction.")
    dummy_img_array = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
    Image.fromarray(dummy_img_array).save(test_image_path)

# Load image
img = cv2.imread(test_image_path)
if img is None:
    print(f"Error: Could not load image from '{test_image_path}'. Check file path and integrity.")
    # As a fallback, create a black image to prevent further errors
    img = np.zeros((128, 128, 3), dtype=np.uint8)

img = cv2.resize(img, (128,128))
img = np.expand_dims(img, axis=0) / 255.0

# Predict
prediction = model.predict(img)
class_index = np.argmax(prediction)

print("Predicted Class:", class_index)

Attempting to load model from: plant_disease_model.keras
'plant_disease_model.keras' exists. File size: 39706393 bytes.
'plant_disease_model.keras' appears to be a valid zip file. Contents: ['metadata.json', 'config.json', 'model.weights.h5']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
Predicted Class: 0
